# 05 扩展内容：PCHIP、B 样条、二维插值与微分矩阵入口

前面几节已经完成了第二章的主线：从插值问题、全局多项式插值、Runge 现象、切比雪夫节点、牛顿差商，到分段线性插值、分段三次 Hermite 插值和自然三次样条。本节继续补充几个后续章节会反复用到的扩展主题：

1. PCHIP 与保形分段三次 Hermite 插值；
2. 三次均匀 B 样条的局部支撑思想；
3. 矩形单元上的双线性插值；
4. 三角形单元上的二维一次 Lagrange 插值；
5. 切比雪夫微分矩阵作为后续谱方法的入口。

这些内容仍然属于第二章“数据插值”的范围，但它们不是孤立的技巧。PCHIP 回答“如何减少三次曲线过冲”，B 样条回答“如何用局部基函数组织光滑曲线”，二维插值回答“如何从一维节点推广到网格和单元”，切比雪夫微分矩阵则说明插值多项式还可以被用来构造数值微分算子。


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
    "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

current = pathlib.Path.cwd().resolve()
PROJECT_ROOT = current if (current / "src" / "py_sc").exists() else None
if PROJECT_ROOT is None:
    for parent in current.parents:
        if (parent / "src" / "py_sc").exists():
            PROJECT_ROOT = parent
            break
if PROJECT_ROOT is None:
    raise RuntimeError("没有找到仓库根目录，请从 py-sc 仓库内运行本 notebook。")

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from py_sc import (  # noqa: E402
    NaturalCubicSpline,
    bilinear_interpolate_cell,
    cubic_uniform_b_spline_basis,
    pchip_interpolate,
    pchip_slopes,
    piecewise_cubic_hermite_interpolate,
    triangle_linear_interpolate,
)


## 1. PCHIP：为什么要限制 Hermite 斜率

分段三次 Hermite 插值在每个节点给定函数值 \(y_i\) 和导数近似 \(m_i\)。对区间 \([x_i,x_{i+1}]\)，设

$$
h_i=x_{i+1}-x_i,\qquad t=\frac{x-x_i}{h_i},
$$

局部插值函数为

$$
H_i(x)=h_{00}(t)y_i+h_{10}(t)h_i m_i
+h_{01}(t)y_{i+1}+h_{11}(t)h_i m_{i+1}.
$$

问题在于：如果节点斜率 \(m_i\) 估计过大，三次曲线可能在两个节点之间冲过数据范围。对单调数据，这种过冲会破坏“数据在上升，插值曲线也应上升”的直观要求。PCHIP（piecewise cubic Hermite interpolating polynomial）的核心思想就是：仍然使用三次 Hermite 形式，但用更谨慎的规则选择节点斜率。

先定义相邻节点的割线斜率

$$
d_i=\frac{y_{i+1}-y_i}{x_{i+1}-x_i}.
$$

如果 \(d_{i-1}\) 和 \(d_i\) 异号，说明 \(x_i\) 附近可能是局部极值。此时把 \(m_i\) 设为 0，可以避免曲线继续沿错误方向冲出。如果二者同号，则用加权调和平均来估计节点斜率：

$$
m_i=
\frac{w_1+w_2}{\dfrac{w_1}{d_{i-1}}+\dfrac{w_2}{d_i}},
\qquad
w_1=2h_i+h_{i-1},\quad
w_2=h_i+2h_{i-1}.
$$

调和平均会偏向较小的斜率，因此比普通算术平均更保守。这正是 PCHIP 减少过冲的关键。


In [ ]:
def teaching_pchip_slopes(x, y):
    """教学版 PCHIP 节点斜率计算。

    这个版本只保留核心逻辑：内部节点用 PCHIP 规则，端点用简单单侧割线。
    src 中的 pchip_slopes 对端点斜率做了更完整的限制，更适合复用。
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    order = np.argsort(x)
    x = x[order]
    y = y[order]

    h = np.diff(x)
    d = np.diff(y) / h
    m = np.zeros_like(y)

    m[0] = d[0]
    m[-1] = d[-1]

    for i in range(1, len(x) - 1):
        if d[i - 1] == 0 or d[i] == 0 or np.sign(d[i - 1]) != np.sign(d[i]):
            m[i] = 0.0
        else:
            w1 = 2 * h[i] + h[i - 1]
            w2 = h[i] + 2 * h[i - 1]
            m[i] = (w1 + w2) / (w1 / d[i - 1] + w2 / d[i])

    return x, y, m

x_data = np.array([0.0, 1.0, 2.0, 3.0, 4.5, 6.0])
y_data = np.array([0.0, 0.8, 0.95, 1.02, 1.08, 1.10])
x_eval = np.linspace(x_data.min(), x_data.max(), 500)

x_sorted, y_sorted, m_teach = teaching_pchip_slopes(x_data, y_data)
_, _, m_src = pchip_slopes(x_data, y_data)

print("教学版内部斜率：", np.round(m_teach, 4))
print("src 版 PCHIP 斜率：", np.round(m_src, 4))
print("二者主要差异在端点处理，内部节点遵循相同保形原则。")


上面的教学版实现故意把端点处理写得很简单，便于看清内部节点的保形规则。`src/py_sc/interpolation.py` 中的 `pchip_slopes` 使用更完整的端点限制：如果端点预测斜率方向和相邻割线方向不一致，就把端点斜率压到 0；如果相邻割线变号且端点斜率过大，则限制到 \(3d_0\) 或 \(3d_{n-1}\) 的范围内。这样可以进一步减少边界附近的过冲。


In [ ]:
# 对比普通中心差分 Hermite、自然三次样条和 PCHIP。
center_slopes = np.zeros_like(y_data)
center_slopes[0] = (y_data[1] - y_data[0]) / (x_data[1] - x_data[0])
center_slopes[-1] = (y_data[-1] - y_data[-2]) / (x_data[-1] - x_data[-2])
center_slopes[1:-1] = (y_data[2:] - y_data[:-2]) / (x_data[2:] - x_data[:-2])

hermite_center = piecewise_cubic_hermite_interpolate(x_data, y_data, center_slopes, x_eval)
pchip_values = pchip_interpolate(x_data, y_data, x_eval)
spline_values = NaturalCubicSpline.fit(x_data, y_data)(x_eval)

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.plot(x_eval, hermite_center, label="中心差分 Hermite", color="#d95f02")
ax.plot(x_eval, spline_values, label="自然三次样条", color="#7570b3")
ax.plot(x_eval, pchip_values, label="PCHIP", color="#1b9e77", linewidth=2.4)
ax.scatter(x_data, y_data, color="black", zorder=3, label="节点")
ax.axhline(y_data.max(), color="gray", linestyle=":", linewidth=1)
ax.set_title("单调数据上的过冲对比")
ax.set_xlabel("x")
ax.set_ylabel("插值值")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

print("PCHIP 是否保持在节点值范围内：", pchip_values.min() >= y_data.min() - 1e-12 and pchip_values.max() <= y_data.max() + 1e-12)
print("PCHIP 是否保持非下降：", np.all(np.diff(pchip_values) >= -1e-10))


从图像可以看到，PCHIP 的目标不是让曲线“尽可能平滑”，而是在保持 \(C^1\) 连续的同时尊重数据形状。自然三次样条有 \(C^2\) 光滑性，但它的全局方程会让某些局部数据影响更远的区域；普通中心差分 Hermite 形式简单，但斜率估计过大时更容易过冲。PCHIP 更适合单调表格数据、物性参数插值、累积分布函数等需要保持形状的场景。


## 2. 三次均匀 B 样条：局部支撑的基函数

三次样条可以用分段多项式和连续性条件来推导，也可以用 B 样条基函数来组织。B 样条的核心优势是**局部支撑**：一个基函数只在有限区间内非零。因此修改一个控制系数时，只会影响附近一小段曲线。

三次均匀 B 样条的一个标准基函数可以用截断幂函数写成

$$
B(x)=\frac{1}{6}\left[(x)_+^3-4(x-1)_+^3+6(x-2)_+^3-4(x-3)_+^3+(x-4)_+^3\right],
$$

其中

$$
(x)_+=\max(x,0).
$$

这个基函数只在 \([0,4]\) 上非零。把它平移得到 \(B(x-k)\)，就可以构造一组局部基函数。


In [ ]:
def teaching_truncated_power_3(x):
    return np.maximum(np.asarray(x, dtype=float), 0.0) ** 3


def teaching_cubic_uniform_b_spline_basis(x):
    x = np.asarray(x, dtype=float)
    return (
        teaching_truncated_power_3(x)
        - 4 * teaching_truncated_power_3(x - 1)
        + 6 * teaching_truncated_power_3(x - 2)
        - 4 * teaching_truncated_power_3(x - 3)
        + teaching_truncated_power_3(x - 4)
    ) / 6.0

x_grid = np.linspace(-1.0, 7.0, 600)
np.testing.assert_allclose(
    teaching_cubic_uniform_b_spline_basis(x_grid),
    cubic_uniform_b_spline_basis(x_grid),
)
print("教学版 B 样条基函数与 src 版实现一致。")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
for shift, color in zip(range(4), ["#1b9e77", "#d95f02", "#7570b3", "#e7298a"]):
    ax.plot(x_grid, cubic_uniform_b_spline_basis(x_grid - shift), label=f"B(x-{shift})", color=color)

ax.set_title("三次均匀 B 样条基函数的局部支撑")
ax.set_xlabel("x")
ax.set_ylabel("基函数值")
ax.legend()
ax.grid(alpha=0.25)
plt.show()


局部支撑使 B 样条和全局多项式插值形成鲜明对比。全局多项式中，改变一个节点值通常会改变整条曲线；B 样条中，每个基函数只覆盖有限区间，因此局部修改更容易控制。后续如果继续深入样条空间，需要引入节点向量、Cox-de Boor 递推公式、控制点和边界处理。本节先保留最重要的直觉：三次 B 样条是一组局部、光滑、可平移的基函数。


## 3. 矩形单元上的双线性插值

一维插值只有一个自变量。二维插值需要根据二维网格或单元结构来构造局部近似。最简单的情形是矩形单元

$$
[x_0,x_1]\times[y_0,y_1].
$$

设四个角点函数值为

$$
z_{00}=f(x_0,y_0),\quad z_{10}=f(x_1,y_0),\quad
z_{01}=f(x_0,y_1),\quad z_{11}=f(x_1,y_1).
$$

令

$$
u=\frac{x-x_0}{x_1-x_0},\qquad
v=\frac{y-y_0}{y_1-y_0}.
$$

双线性插值写成

$$
F(u,v)=(1-u)(1-v)z_{00}+u(1-v)z_{10}+(1-u)vz_{01}+uvz_{11}.
$$

它对每个方向分别是线性的，但整体包含 \(uv\) 项，所以通常不是关于 \((x,y)\) 的二维一次函数。双线性插值常用于规则网格数据、图像缩放和二维查表。


In [ ]:
def teaching_bilinear_cell(x_bounds, y_bounds, values, x_eval, y_eval):
    x0, x1 = x_bounds
    y0, y1 = y_bounds
    values = np.asarray(values, dtype=float)
    x_eval = np.asarray(x_eval, dtype=float)
    y_eval = np.asarray(y_eval, dtype=float)

    u = (x_eval - x0) / (x1 - x0)
    v = (y_eval - y0) / (y1 - y0)
    z00, z10 = values[0, 0], values[0, 1]
    z01, z11 = values[1, 0], values[1, 1]
    return (1 - u) * (1 - v) * z00 + u * (1 - v) * z10 + (1 - u) * v * z01 + u * v * z11

x_bounds = (0.0, 1.0)
y_bounds = (0.0, 1.0)
corner_values = np.array([[0.0, 1.0], [1.2, 0.2]])

u = np.linspace(0.0, 1.0, 80)
v = np.linspace(0.0, 1.0, 80)
X, Y = np.meshgrid(u, v)
Z_teach = teaching_bilinear_cell(x_bounds, y_bounds, corner_values, X, Y)
Z_src = bilinear_interpolate_cell(x_bounds, y_bounds, corner_values, X, Y)
np.testing.assert_allclose(Z_teach, Z_src)
print("教学版双线性插值与 src 版实现一致。")


In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 5.2))
contour = ax.contourf(X, Y, Z_src, levels=20, cmap="viridis")
ax.scatter([0, 1, 0, 1], [0, 0, 1, 1], color="white", edgecolor="black", s=70, zorder=3)
for x0, y0, label in [(0, 0, "z00"), (1, 0, "z10"), (0, 1, "z01"), (1, 1, "z11")]:
    ax.text(x0, y0, label, ha="center", va="bottom", color="black", fontsize=11)
ax.set_title("矩形单元上的双线性插值")
ax.set_xlabel("x")
ax.set_ylabel("y")
fig.colorbar(contour, ax=ax, label="插值值")
plt.show()


双线性插值可以看作两次一维线性插值：先在 \(x\) 方向对下边和上边分别插值，再在 \(y\) 方向插值。这种构造简单、局部、稳定，但只保证单元内部的低阶近似；跨单元拼接时通常只能保证函数值连续，不保证梯度连续。


## 4. 三角形单元上的二维一次 Lagrange 插值

二维网格不一定是矩形。有限元方法中常用三角形单元。设三角形三个顶点为

$$
P_1=(x_1,y_1),\quad P_2=(x_2,y_2),\quad P_3=(x_3,y_3),
$$

任意位于三角形内的点 \(P=(x,y)\) 可以写成重心坐标形式

$$
P=\lambda_1P_1+\lambda_2P_2+\lambda_3P_3,
\qquad
\lambda_1+\lambda_2+\lambda_3=1.
$$

如果顶点函数值为 \(f_1,f_2,f_3\)，二维一次 Lagrange 插值就是

$$
F(P)=\lambda_1 f_1+\lambda_2 f_2+\lambda_3 f_3.
$$

这里的 \(\lambda_i\) 就是二维一次 Lagrange 基函数。它们分别在自己的顶点取 1，在其他顶点取 0，和一维拉格朗日基函数的“开关函数”思想完全一致。


In [ ]:
def teaching_barycentric_coordinates(points, vertices):
    points = np.asarray(points, dtype=float)
    vertices = np.asarray(vertices, dtype=float)
    matrix = np.array(
        [
            [vertices[0, 0], vertices[1, 0], vertices[2, 0]],
            [vertices[0, 1], vertices[1, 1], vertices[2, 1]],
            [1.0, 1.0, 1.0],
        ],
        dtype=float,
    )
    flat_points = points.reshape(-1, 2)
    rhs = np.vstack([flat_points.T, np.ones(flat_points.shape[0])])
    lambdas = np.linalg.solve(matrix, rhs).T
    return lambdas.reshape(points.shape[:-1] + (3,))

vertices = np.array([[0.0, 0.0], [1.4, 0.2], [0.3, 1.1]])
vertex_values = 1.0 + 2.0 * vertices[:, 0] - 0.7 * vertices[:, 1]
test_points = np.array([[0.35, 0.25], [0.55, 0.55]])

lambdas = teaching_barycentric_coordinates(test_points, vertices)
teaching_values = lambdas @ vertex_values
src_values = triangle_linear_interpolate(vertices, vertex_values, test_points)
np.testing.assert_allclose(teaching_values, src_values)
print("测试点的重心坐标：")
print(np.round(lambdas, 4))
print("二维一次插值值：", np.round(src_values, 4))


In [ ]:
# 在三角形内部采样并画出一次插值曲面在平面上的等值线。
grid_x = np.linspace(vertices[:, 0].min(), vertices[:, 0].max(), 140)
grid_y = np.linspace(vertices[:, 1].min(), vertices[:, 1].max(), 140)
GX, GY = np.meshgrid(grid_x, grid_y)
grid_points = np.stack([GX, GY], axis=-1)
grid_lambdas = teaching_barycentric_coordinates(grid_points, vertices)
inside = np.all(grid_lambdas >= -1e-12, axis=-1)
GZ = np.full(GX.shape, np.nan)
GZ[inside] = triangle_linear_interpolate(vertices, vertex_values, grid_points[inside])

fig, ax = plt.subplots(figsize=(6.4, 5.2))
contour = ax.contourf(GX, GY, GZ, levels=18, cmap="magma")
ax.plot(
    [vertices[0, 0], vertices[1, 0], vertices[2, 0], vertices[0, 0]],
    [vertices[0, 1], vertices[1, 1], vertices[2, 1], vertices[0, 1]],
    color="white",
    linewidth=2,
)
ax.scatter(vertices[:, 0], vertices[:, 1], color="white", edgecolor="black", s=80, zorder=3)
ax.set_title("三角形单元上的二维一次插值")
ax.set_xlabel("x")
ax.set_ylabel("y")
fig.colorbar(contour, ax=ax, label="插值值")
plt.show()


三角形一次插值在单元内部给出一个平面，因此它能精确表示任何二维一次函数。它的优势是几何适应性强，可以处理不规则区域；缺点是单元内近似阶数较低，跨单元的梯度通常不连续。后续学习有限元方法时，重心坐标和三角形 Lagrange 基函数会再次出现。


## 5. 切比雪夫微分矩阵：从插值到数值微分

切比雪夫微分矩阵不是第二章的主线，但它能说明插值思想如何进入后续谱方法。给定切比雪夫配置点

$$
x_j=\cos\frac{j\pi}{N},\qquad j=0,1,\ldots,N,
$$

先构造拉格朗日插值多项式

$$
p_N(x)=\sum_{j=0}^{N} f(x_j)\ell_j(x).
$$

对它求导并在节点 \(x_i\) 上取值：

$$
p_N'(x_i)=\sum_{j=0}^{N} f(x_j)\ell_j'(x_i).
$$

于是定义

$$
D_{ij}=\ell_j'(x_i),
$$

就得到微分矩阵

$$
\mathbf f'\approx D\mathbf f.
$$

这条逻辑链是：节点函数值 \(\rightarrow\) 插值多项式 \(\rightarrow\) 对插值多项式求导 \(\rightarrow\) 微分矩阵。本节只给出可运行入口，完整的稳定公式、边界处理和谱精度分析应放在后续“谱方法”章节。


In [ ]:
def teaching_lagrange_derivative_matrix(nodes):
    """直接对 Lagrange 基函数求导，构造小规模微分矩阵。

    这是教学实现，复杂度较高，只适合解释 D_ij = l_j'(x_i) 的定义。
    后续谱方法章节会使用更稳定的闭式公式。
    """
    nodes = np.asarray(nodes, dtype=float)
    n = len(nodes)
    D = np.zeros((n, n), dtype=float)

    for i, xi in enumerate(nodes):
        for j, xj in enumerate(nodes):
            total = 0.0
            for m in range(n):
                if m == j:
                    continue
                term = 1.0 / (xj - nodes[m])
                for r in range(n):
                    if r == j or r == m:
                        continue
                    term *= (xi - nodes[r]) / (xj - nodes[r])
                total += term
            D[i, j] = total
    return D

N = 10
cheb_nodes = np.cos(np.arange(N + 1) * np.pi / N)
D = teaching_lagrange_derivative_matrix(cheb_nodes)

f_values = np.exp(cheb_nodes)
derivative_approx = D @ f_values
derivative_exact = np.exp(cheb_nodes)
max_error = np.max(np.abs(derivative_approx - derivative_exact))
print("N =", N)
print("微分矩阵近似 exp(x) 导数的最大误差：", f"{max_error:.3e}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
order = np.argsort(cheb_nodes)
ax.plot(cheb_nodes[order], derivative_exact[order], label="真实导数 exp(x)", color="#1b9e77")
ax.scatter(cheb_nodes[order], derivative_approx[order], label="D f 的节点值", color="#d95f02", zorder=3)
ax.set_title("由插值多项式导数得到的节点导数近似")
ax.set_xlabel("x")
ax.set_ylabel("导数值")
ax.legend()
ax.grid(alpha=0.25)
plt.show()


## 小结

本节把第二章尚未展开的几个主题补成了可运行入口。

* PCHIP 仍然属于分段三次 Hermite 插值，但通过限制节点斜率来保持单调性并减少过冲。
* 三次均匀 B 样条强调局部支撑，它是后续更一般样条空间和曲线设计的基础。
* 双线性插值把一维线性插值推广到矩形单元，适合规则网格数据。
* 三角形一次插值用重心坐标构造二维 Lagrange 基函数，是有限元方法的重要入口。
* 切比雪夫微分矩阵说明：插值多项式不仅能近似函数值，也能通过求导构造数值微分算子。

这些内容中，PCHIP、B 样条和二维插值已经可以作为第二章的扩展阅读；切比雪夫微分矩阵只保留入门连接，后续应放在谱方法章节系统展开。
